## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-02-14-51-00 +0000,wsj,Oil Prices Fall More Than 5% as Iran Peace Tal...,https://www.wsj.com/world/middle-east/oil-pric...
1,2026-11-02-14-50-40 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/02/us/mid...
2,2026-11-02-14-48-59 +0000,nypost,Victim of doomed Florida plane heard reciting ...,https://nypost.com/2026/11/02/us-news/victim-o...
3,2026-11-02-14-47-09 +0000,nyt,Trump Skirts Court Commands as Distrust Betwee...,https://www.nytimes.com/2026/11/02/us/trump-co...
4,2026-11-02-14-45-21 +0000,bbc,Labour ends power-sharing with Greens in 11 co...,https://www.bbc.co.uk/news/articles/c63dj21yvj...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2650/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
24,trump,19
103,senate,18
102,race,17
12,florida,14
219,election,13


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
162,2026-11-02-05-08-00 +0000,wsj,Rep. Haley Stevens stunned Democrats on Sunday...,https://www.wsj.com/politics/elections/how-rep...,146
245,2026-11-01-18-25-38 +0000,chitri,Democrat Haley Stevens endorses GOP’s Mike Rog...,https://www.chicagotribune.com/2026/11/01/demo...,107
254,2026-11-01-17-37-01 +0000,nypost,Abdul El-Sayed’s ex-Dem opponent Rep. Haley St...,https://nypost.com/2026/11/01/us-news/abdul-el...,92
169,2026-11-02-03-10-02 +0000,startribune,El-Sayed's primary opponent stuns Democrats by...,https://www.startribune.com/michigan-democrats...,91
201,2026-11-01-22-44-56 +0000,nypost,Michigan Senate candidate Mike Rogers says he’...,https://nypost.com/2026/11/01/us-news/michigan...,90


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
162,146,2026-11-02-05-08-00 +0000,wsj,Rep. Haley Stevens stunned Democrats on Sunday...,https://www.wsj.com/politics/elections/how-rep...
68,79,2026-11-02-11-55-00 +0000,wsj,As the midterms bumble and stumble and fumble ...,https://www.wsj.com/politics/trumps-monday-spe...
230,62,2026-11-01-20-00-00 +0000,wsj,"In the Florida governor’s race, Democrat David...",https://www.wsj.com/politics/elections/democra...
196,42,2026-11-01-23-43-49 +0000,bbc,Judge dismisses Trump administration's lawsui...,https://www.bbc.co.uk/news/articles/cv4g1y02je...
124,40,2026-11-02-10-00-00 +0000,nypost,Trump allies brace for massive legal bills if ...,https://nypost.com/2026/11/02/us-news/trump-al...
146,38,2026-11-02-07-45-39 +0000,nypost,Crazed University of Michigan student pushes c...,https://nypost.com/2026/11/02/us-news/universi...
0,34,2026-11-02-14-51-00 +0000,wsj,Oil Prices Fall More Than 5% as Iran Peace Tal...,https://www.wsj.com/world/middle-east/oil-pric...
49,34,2026-11-02-12-35-00 +0000,wsj,The Morning Risk Report: Binance Resumes Withd...,https://www.wsj.com/risk-compliance-journal/th...
128,31,2026-11-02-09-36-45 +0000,bbc,Martha Stewart says she 'really screwed up' wi...,https://www.bbc.co.uk/news/articles/c8zxle6qq4...
73,31,2026-11-02-11-52-19 +0000,cbc,Canadian among the dead in Spanish high-speed ...,https://www.cbc.ca/news/world/spain-officials-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
